# S02 — Sync an Ubuntu Data Lake Directory to Amazon S3

This notebook synchronizes the `datalake` directory in the Ubuntu user's home directory (`~/datalake`) to an existing S3 bucket in `us-east-1`. It uses the AWS CLI profile named `training`.

The bucket name is read from the `S3_BUCKET_NAME` environment variable. If that variable is not set, the notebook uses `gksdatalake`.

## 1. Prerequisites

Before continuing:

- AWS CLI v2 must be installed.
- The `training` profile must be configured.
- The destination S3 bucket must already exist.
- The profile must have permission to list the bucket and upload objects.
- The local directory `~/datalake` must exist.

In [ ]:
%%bash
set -euo pipefail
aws --version
aws sts get-caller-identity --profile training
aws configure get region --profile training

## 2. Set or inspect the bucket name

To use a bucket other than `gksdatalake`, set the environment variable before starting Jupyter, change the bucket name to yours: we can add to .bashrc and jupyter settings.

```bash
export S3_BUCKET_NAME=my-existing-bucket
jupyter lab
```

The shell expressions below use `gksdatalake` only when `S3_BUCKET_NAME` is unset or empty.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
LOCAL_DATALAKE="$HOME/datalake"
printf 'Local source: %s\nS3 destination: s3://%s/\n' "$LOCAL_DATALAKE" "$S3_BUCKET_NAME"

## 3. Verify the local directory and S3 bucket

This cell fails early if the local directory is missing or the bucket cannot be accessed. It does not modify local files or S3 objects.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
LOCAL_DATALAKE="$HOME/datalake"

if [[ ! -d "$LOCAL_DATALAKE" ]]; then
  echo "Directory not found: $LOCAL_DATALAKE" >&2
  exit 1
fi

find "$LOCAL_DATALAKE" -maxdepth 2 -type f | sed -n '1,20p'
aws s3api head-bucket --bucket "$S3_BUCKET_NAME" --profile training
echo "Ready to sync to s3://$S3_BUCKET_NAME/"

## 4. Preview the sync

Always run with `--dryrun` first. It shows which files would be uploaded without changing S3. The trailing slash means the contents of `~/datalake` are copied to the root of the bucket.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3 sync "$HOME/datalake/" "s3://$S3_BUCKET_NAME/" \
  --dryrun \
  --region us-east-1 \
  --profile training

## 5. Sync `~/datalake` to S3

Run this cell after reviewing the dry-run output. By default, `aws s3 sync` uploads new and changed local files. It does not remove extra files already in S3 because `--delete` is intentionally not used.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3 sync "$HOME/datalake/" "s3://$S3_BUCKET_NAME/" \
  --region us-east-1 \
  --profile training

## 6. Verify uploaded objects

List the synchronized objects and display the total object count and size.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3 ls "s3://$S3_BUCKET_NAME/" \
  --recursive \
  --human-readable \
  --summarize \
  --region us-east-1 \
  --profile training

## Sync command summary

The essential Ubuntu command is:

```bash
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3 sync "$HOME/datalake/" "s3://$S3_BUCKET_NAME/" --region us-east-1 --profile training
```